# Inserting data into a database in SQL

Inserting data into a database is one of the fundamental operations in SQL. The `INSERT` statement in MySQL allows you to add new rows to a table. Depending on the context and requirements, there are several ways to insert data. Here's an in-depth explanation:

### **1. Basic INSERT Statement**

The most straightforward way to insert data into a table is by using the `INSERT INTO` statement. This can be done in two main ways: specifying the columns or without specifying the columns.

#### **Syntax:**
```sql
INSERT INTO table_name (column1, column2, column3, ...)
VALUES (value1, value2, value3, ...);
```

#### **Example:**
Suppose you have a `Customers` table with columns `CustomerID`, `CustomerName`, `Country`:

```sql
INSERT INTO Customers (CustomerID, CustomerName, Country)
VALUES (1, 'John Doe', 'USA');
```

In this example:
- The `INSERT INTO` statement is used to specify the table (`Customers`) into which data is to be inserted.
- The columns (`CustomerID`, `CustomerName`, `Country`) where the data will be inserted are specified.
- The `VALUES` clause provides the actual data to be inserted.

#### **Omitting Column Names:**
If you omit the column names, you must provide values for every column in the table, in the order the columns are defined in the table schema.

```sql
INSERT INTO Customers
VALUES (2, 'Jane Smith', 'Canada');
```

This method is less recommended because it tightly couples your query to the exact structure of the table, which could lead to errors if the table structure changes.

### **2. Inserting Multiple Rows**

You can insert multiple rows in a single `INSERT` statement by separating each set of values with a comma.

#### **Syntax:**
```sql
INSERT INTO table_name (column1, column2, column3, ...)
VALUES 
    (value1a, value2a, value3a, ...),
    (value1b, value2b, value3b, ...),
    ...;
```

#### **Example:**
```sql
INSERT INTO Customers (CustomerID, CustomerName, Country)
VALUES 
    (3, 'Alice Johnson', 'UK'),
    (4, 'Robert Brown', 'Australia');
```

This inserts two rows into the `Customers` table.

### **3. Inserting Data with SELECT**

You can insert data into a table by selecting data from one or more other tables. This is useful for copying data from one table to another.

#### **Syntax:**
```sql
INSERT INTO table_name (column1, column2, column3, ...)
SELECT column1, column2, column3, ...
FROM other_table
WHERE condition;
```

#### **Example:**
Assume you have a `Customers_Backup` table, and you want to copy all customers from `Germany` in the `Customers` table to this backup table:

```sql
INSERT INTO Customers_Backup (CustomerID, CustomerName, Country)
SELECT CustomerID, CustomerName, Country
FROM Customers
WHERE Country = 'Germany';
```

This will insert all rows from `Customers` where `Country` is `'Germany'` into the `Customers_Backup` table.

### **4. Inserting with Default Values**

When you insert data, you can specify default values for certain columns by either omitting them or explicitly using the `DEFAULT` keyword.

#### **Example:**
Assume the `Customers` table has a `CreatedDate` column with a default value of the current date:

```sql
INSERT INTO Customers (CustomerID, CustomerName, Country)
VALUES (5, 'George White', 'France');
```

Or, explicitly:

```sql
INSERT INTO Customers (CustomerID, CustomerName, Country, CreatedDate)
VALUES (6, 'Henry Black', 'Spain', DEFAULT);
```

### **5. Inserting NULL Values**

If a column can accept `NULL` values, you can insert a `NULL` into that column either by not including it in the list of columns or by explicitly inserting `NULL`.

#### **Example:**
```sql
INSERT INTO Customers (CustomerID, CustomerName, Country)
VALUES (7, 'Emma Blue', NULL);
```

This inserts a `NULL` for the `Country` column.

### **6. Inserting Data with AUTO_INCREMENT Columns**

For tables with an `AUTO_INCREMENT` column (often a primary key), you don’t need to specify a value for that column. MySQL will automatically generate the next sequential value.

#### **Example:**
If `CustomerID` is an `AUTO_INCREMENT` column:

```sql
INSERT INTO Customers (CustomerName, Country)
VALUES ('Liam Green', 'Ireland');
```

MySQL will automatically generate the `CustomerID`.

### **7. Handling Duplicates with INSERT**

Sometimes, you might want to insert a row only if it does not already exist, or update it if it does. MySQL offers two options for handling this scenario:

#### **a. INSERT IGNORE:**
This statement ignores the insert operation if a duplicate key error occurs (e.g., trying to insert a row with a primary key that already exists).

```sql
INSERT IGNORE INTO Customers (CustomerID, CustomerName, Country)
VALUES (1, 'John Doe', 'USA');
```

If a row with `CustomerID = 1` already exists, MySQL will ignore this insert without throwing an error.

#### **b. ON DUPLICATE KEY UPDATE:**
This statement allows you to update the row if a duplicate key is found.

```sql
INSERT INTO Customers (CustomerID, CustomerName, Country)
VALUES (1, 'John Doe', 'USA')
ON DUPLICATE KEY UPDATE 
    CustomerName = VALUES(CustomerName),
    Country = VALUES(Country);
```

If a row with `CustomerID = 1` exists, this query will update `CustomerName` and `Country` instead of inserting a new row.

### **8. Using Stored Procedures to Insert Data**

For more complex scenarios, you can use stored procedures to insert data. Stored procedures allow you to encapsulate complex logic within the database.

#### **Example:**
```sql
DELIMITER //
CREATE PROCEDURE AddCustomer(
    IN p_CustomerName VARCHAR(100),
    IN p_Country VARCHAR(50)
)
BEGIN
    INSERT INTO Customers (CustomerName, Country)
    VALUES (p_CustomerName, p_Country);
END //
DELIMITER ;
```

You can call this procedure as follows:

```sql
CALL AddCustomer('Sophia Red', 'Italy');
```

This procedure will insert a new customer with the provided name and country.

### **9. Inserting Data with Transactions**

When inserting data that involves multiple steps or affects multiple tables, you may want to use transactions to ensure data integrity. Transactions allow you to group multiple SQL statements into a single unit of work that either completes entirely or not at all.

#### **Example:**
```sql
START TRANSACTION;

INSERT INTO Orders (OrderID, OrderDate, CustomerID)
VALUES (1, '2024-08-10', 1);

INSERT INTO OrderDetails (OrderID, ProductID, Quantity)
VALUES (1, 101, 2);

COMMIT;
```

If any part of the transaction fails, you can use `ROLLBACK` to undo all changes:

```sql
ROLLBACK;
```

### **10. Bulk Inserts**

When you need to insert a large amount of data, you might consider using bulk inserts to improve performance. You can insert multiple rows at once as shown in the "Inserting Multiple Rows" section, or use tools like `LOAD DATA INFILE` for large-scale data imports.

#### **Example with `LOAD DATA INFILE`:**

```sql
LOAD DATA INFILE '/path/to/your/file.csv'
INTO TABLE Customers
FIELDS TERMINATED BY ',' 
LINES TERMINATED BY '\n'
(CustomerID, CustomerName, Country);
```

### **Conclusion**

Inserting data in MySQL is a versatile operation with many options to accommodate different scenarios. From basic inserts to handling duplicates, using transactions, and bulk operations, understanding these techniques is crucial for efficiently managing and manipulating your data in a MySQL database.

# The DECLARE statement in MySQL stored procedures

Stored procedures in MySQL are a powerful feature that allows you to encapsulate SQL logic within the database, enabling you to perform complex operations, such as inserting data, with a single call. They are useful for ensuring consistent data operations, improving performance, and reducing the need for repeated SQL code in applications.

### **1. What is a Stored Procedure?**

A stored procedure is a set of SQL statements that you can save and reuse. Once created, you can call the procedure multiple times, passing parameters to customize its behavior. Stored procedures are stored in the database and executed on the server side.

### **2. Why Use Stored Procedures for Inserting Data?**

- **Consistency:** Stored procedures ensure that the same logic is applied every time you insert data.
- **Reusability:** You can write the procedure once and call it from multiple places, reducing code duplication.
- **Performance:** Stored procedures are precompiled, which can improve performance, especially when complex logic is involved.
- **Security:** You can restrict direct access to tables and require users to interact with the data through stored procedures, adding a layer of security.

### **3. Creating a Stored Procedure to Insert Data**

Let's break down the process of creating and using a stored procedure for inserting data.

#### **Step 1: Define the Stored Procedure**

To create a stored procedure in MySQL, you use the `CREATE PROCEDURE` statement. You need to specify:
- The name of the procedure.
- The parameters it will accept (if any).
- The SQL statements that will be executed when the procedure is called.

##### **Syntax:**
```sql
CREATE PROCEDURE procedure_name (parameter_list)
BEGIN
    -- SQL statements go here
END;
```

#### **Step 2: Creating a Simple Insert Stored Procedure**

Let's say you have a `Customers` table, and you want to create a stored procedure to insert a new customer.

##### **Example:**
```sql
DELIMITER //

CREATE PROCEDURE AddCustomer(
    IN p_CustomerName VARCHAR(100),
    IN p_Country VARCHAR(50)
)
BEGIN
    INSERT INTO Customers (CustomerName, Country)
    VALUES (p_CustomerName, p_Country);
END //

DELIMITER ;
```

- **`DELIMITER //`**: This changes the statement delimiter from `;` to `//` to allow for multiple SQL statements in the procedure body.
- **`IN p_CustomerName VARCHAR(100)`**: Defines an input parameter `p_CustomerName` of type `VARCHAR(100)`. The `IN` keyword specifies that this parameter is an input to the procedure.
- **`INSERT INTO Customers (CustomerName, Country)`**: This SQL statement inserts a new customer into the `Customers` table, using the values passed to the procedure.

#### **Step 3: Calling the Stored Procedure**

After creating the stored procedure, you can call it to insert data.

##### **Syntax:**
```sql
CALL procedure_name(parameter_values);
```

##### **Example:**
```sql
CALL AddCustomer('Sophia Red', 'Italy');
```

This command inserts a new customer with the name `'Sophia Red'` and the country `'Italy'` into the `Customers` table.

### **4. Handling More Complex Logic**

Stored procedures can include more complex logic, such as conditional statements (`IF`, `CASE`), loops (`WHILE`, `LOOP`), and error handling.

#### **Example with Conditional Logic:**

Suppose you want to check if a customer already exists before inserting them.

```sql
DELIMITER //

CREATE PROCEDURE AddCustomerIfNotExists(
    IN p_CustomerName VARCHAR(100),
    IN p_Country VARCHAR(50)
)
BEGIN
    DECLARE existing_customer INT;

    -- Check if customer already exists
    SELECT COUNT(*) INTO existing_customer
    FROM Customers
    WHERE CustomerName = p_CustomerName AND Country = p_Country;

    -- If customer does not exist, insert new record
    IF existing_customer = 0 THEN
        INSERT INTO Customers (CustomerName, Country)
        VALUES (p_CustomerName, p_Country);
    END IF;
END //

DELIMITER ;
```

- **`DECLARE existing_customer INT;`**: This declares a local variable `existing_customer` to store the result of the customer count.
- **`SELECT COUNT(*) INTO existing_customer ...`**: This checks if a customer with the same name and country already exists.
- **`IF existing_customer = 0 THEN ...`**: This conditional statement checks if no matching customer exists (`existing_customer = 0`), and if so, inserts the new customer.

#### **Calling the Procedure:**

```sql
CALL AddCustomerIfNotExists('Sophia Red', 'Italy');
```

This call will insert `'Sophia Red'` only if she doesn’t already exist in the `Customers` table.

### **5. Error Handling in Stored Procedures**

Stored procedures can also include error-handling mechanisms to manage potential issues that may arise during execution.

#### **Example with Error Handling:**

```sql
DELIMITER //

CREATE PROCEDURE AddCustomerSafely(
    IN p_CustomerName VARCHAR(100),
    IN p_Country VARCHAR(50)
)
BEGIN
    DECLARE EXIT HANDLER FOR SQLEXCEPTION
    BEGIN
        -- Rollback in case of error
        ROLLBACK;
        SELECT 'An error occurred, operation rolled back.';
    END;

    START TRANSACTION;
    
    -- Insert the customer
    INSERT INTO Customers (CustomerName, Country)
    VALUES (p_CustomerName, p_Country);

    COMMIT;
END //

DELIMITER ;
```

- **`DECLARE EXIT HANDLER FOR SQLEXCEPTION`**: This sets up an error handler that triggers when an SQL exception occurs.
- **`START TRANSACTION;`**: Begins a transaction. If the insert fails, the transaction can be rolled back.
- **`ROLLBACK;`**: Reverts any changes made during the transaction if an error occurs.
- **`COMMIT;`**: Commits the transaction if no errors occur.

#### **Calling the Procedure:**

```sql
CALL AddCustomerSafely('Liam Green', 'Ireland');
```

This procedure inserts the customer into the `Customers` table and ensures that the operation is rolled back if an error occurs.

### **6. Managing Stored Procedures**

You can view, alter, or drop stored procedures as needed:

- **Viewing a Procedure:** Use `SHOW CREATE PROCEDURE procedure_name;` to see the definition of a stored procedure.
- **Altering a Procedure:** To modify a stored procedure, you’ll need to drop it and recreate it, as MySQL doesn’t support the `ALTER PROCEDURE` statement.
- **Dropping a Procedure:** Use `DROP PROCEDURE procedure_name;` to remove a stored procedure.

### **7. Best Practices for Stored Procedures**

- **Parameter Naming:** Use clear, consistent naming conventions for parameters (e.g., prefix with `p_`).
- **Error Handling:** Always include error-handling logic, especially when performing critical operations like inserts.
- **Transactions:** Use transactions to ensure data integrity, particularly when multiple operations are involved.
- **Documentation:** Document your procedures well, as they are often used by multiple applications and developers.

### **8. Advantages and Disadvantages**

#### **Advantages:**
- **Encapsulation:** Encapsulate complex logic in a reusable way.
- **Performance:** Precompiled execution can improve performance.
- **Security:** Restrict direct table access, enforcing the use of stored procedures.

#### **Disadvantages:**
- **Complexity:** Debugging stored procedures can be more difficult than debugging application code.
- **Maintenance:** Changes to procedures require altering the database schema, which can be more cumbersome than updating application logic.

### **Conclusion**

Stored procedures offer a powerful way to manage data insertion and other operations in MySQL. By encapsulating SQL logic, you ensure consistency, reusability, and security in your database interactions. While they require careful planning and management, the benefits they provide, especially in complex or repetitive tasks, can be substantial.

# Stored procedures in MySQL 

The `DECLARE` statement in MySQL stored procedures is used to define local variables, condition handlers, and cursors that are only available within the scope of the procedure or function. Here's a breakdown of how `DECLARE` works and when you use it:

### **1. Declaring Variables**

In MySQL stored procedures, you can use `DECLARE` to define local variables that store temporary data for use within the procedure. These variables are only accessible within the procedure and cease to exist once the procedure finishes executing.

#### **Syntax:**
```sql
DECLARE variable_name datatype [DEFAULT value];
```

- **`variable_name`**: The name of the variable.
- **`datatype`**: The data type of the variable (e.g., `INT`, `VARCHAR`, `DATE`).
- **`DEFAULT value`**: Optional. Specifies the default value for the variable if not explicitly set.

#### **Example:**
```sql
DELIMITER //

CREATE PROCEDURE CalculateDiscount(IN p_Price DECIMAL(10,2), OUT p_Discount DECIMAL(10,2))
BEGIN
    DECLARE discount_rate DECIMAL(5,2) DEFAULT 0.10;
    DECLARE discount_amount DECIMAL(10,2);

    -- Calculate discount
    SET discount_amount = p_Price * discount_rate;
    SET p_Discount = discount_amount;
END //

DELIMITER ;
```

In this example:
- `discount_rate` is declared as a `DECIMAL` with a default value of `0.10`.
- `discount_amount` is declared as a `DECIMAL` to store the calculated discount.

### **2. Declaring Condition Handlers**

A condition handler is a block of code that is executed when a specific condition occurs, such as an error or a warning. You declare a condition handler using `DECLARE ... HANDLER`.

#### **Syntax:**
```sql
DECLARE handler_type HANDLER FOR condition_type action;
```

- **`handler_type`**: Specifies whether the handler is for an `EXIT`, `CONTINUE`, or `UNDO` action.
- **`condition_type`**: The condition that triggers the handler, such as `SQLEXCEPTION`, `SQLWARNING`, or specific error codes.
- **`action`**: The action to take when the condition occurs, such as executing a SQL statement or returning an error message.

#### **Example:**
```sql
DELIMITER //

CREATE PROCEDURE SafeInsertCustomer(IN p_CustomerName VARCHAR(100), IN p_Country VARCHAR(50))
BEGIN
    DECLARE EXIT HANDLER FOR SQLEXCEPTION
    BEGIN
        -- Handle error, rollback transaction
        ROLLBACK;
        SELECT 'Error occurred, transaction rolled back.';
    END;

    START TRANSACTION;
    INSERT INTO Customers (CustomerName, Country) VALUES (p_CustomerName, p_Country);
    COMMIT;
END //

DELIMITER ;
```

In this example, the `DECLARE EXIT HANDLER FOR SQLEXCEPTION` handles any SQL exception by rolling back the transaction and displaying an error message.

### **3. Declaring Cursors**

Cursors are used to retrieve and manipulate the result set of a query row by row. You declare a cursor within a stored procedure using `DECLARE ... CURSOR`.

#### **Syntax:**
```sql
DECLARE cursor_name CURSOR FOR select_statement;
```

- **`cursor_name`**: The name of the cursor.
- **`select_statement`**: The SQL query whose result set will be traversed.

#### **Example:**
```sql
DELIMITER //

CREATE PROCEDURE ListCustomersByCountry(IN p_Country VARCHAR(50))
BEGIN
    DECLARE done INT DEFAULT 0;
    DECLARE customer_name VARCHAR(100);
    DECLARE cur CURSOR FOR 
        SELECT CustomerName FROM Customers WHERE Country = p_Country;

    DECLARE CONTINUE HANDLER FOR NOT FOUND SET done = 1;

    OPEN cur;

    read_loop: LOOP
        FETCH cur INTO customer_name;
        IF done THEN
            LEAVE read_loop;
        END IF;
        -- Process each customer_name
        SELECT customer_name;
    END LOOP;

    CLOSE cur;
END //

DELIMITER ;
```

In this example:
- **`cur`** is declared as a cursor for a `SELECT` statement that retrieves customers by country.
- **`done`** is a control variable to manage the loop’s exit condition.
- **`DECLARE CONTINUE HANDLER FOR NOT FOUND`** handles the end of the cursor by setting `done` to `1`.

### **4. Rules and Limitations**

- **Order of Declaration:** All `DECLARE` statements must appear at the beginning of the procedure block, before any executable statements (such as `SELECT`, `INSERT`, etc.).
- **Scope:** Variables, handlers, and cursors declared with `DECLARE` are local to the stored procedure or function and are not accessible outside of it.
- **Default Values:** If a variable is declared without a default value, it is initialized to `NULL`.

### **Conclusion**

The `DECLARE` statement is essential in MySQL stored procedures for creating local variables, defining condition handlers, and managing cursors. It enables the handling of complex logic, improves code readability, and provides the means to control and handle errors effectively. Understanding how to use `DECLARE` properly is key to writing robust and maintainable stored procedures.